In [1]:
print("hello")

hello


In [2]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset
import torchvision.transforms as T

class LeafDataset(Dataset):
    def __init__(self, image_dir, mask_dir):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.files = os.listdir(image_dir)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        img_path = os.path.join(self.image_dir, file)
        mask_path = os.path.join(self.mask_dir, file)

        # Load Image
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Image not found: {img_path}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Load Mask
        mask = cv2.imread(mask_path, 0)
        if mask is None:
            raise ValueError(f"Mask not found: {mask_path}")

        # Convert to Binary Mask
        masks = (mask > 0).astype(np.uint8)

        pos = np.where(masks)

        if len(pos[0]) == 0 or len(pos[1]) == 0:
            raise ValueError(f"Empty mask found: {file}")

        xmin = np.min(pos[1])
        xmax = np.max(pos[1])
        ymin = np.min(pos[0])
        ymax = np.max(pos[0])

        # Validate Bounding Box
        if xmax <= xmin or ymax <= ymin:
            raise ValueError(
                f"Invalid bbox in {file}: [{xmin},{ymin},{xmax},{ymax}]"
            )
        boxes = torch.as_tensor([[xmin, ymin, xmax, ymax]], dtype=torch.float32)
        labels = torch.as_tensor([1], dtype=torch.int64)
        masks = torch.as_tensor(masks[np.newaxis, :, :], dtype=torch.uint8)

        area = (xmax - xmin) * (ymax - ymin)

        if area <= 0:
            raise ValueError(f"Invalid area in {file}")

        target = {
            "boxes": boxes,
            "labels": labels,
            "masks": masks,
            "image_id": torch.tensor([idx]),
            "area": torch.tensor([area], dtype=torch.float32),
            "iscrowd": torch.zeros((1,), dtype=torch.int64)
        }
       
        transform = T.Compose([
            T.ToTensor(),
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.3),
            T.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2
            )
        ])

        img = transform(img)
        

        return img, target

In [3]:
import os

image_dir = r"E:\demo dataset\images"
mask_dir = r"E:\demo dataset\binary_masks"

for file in os.listdir(image_dir):
    if not os.path.exists(os.path.join(mask_dir, file)):
        print("Missing Mask:", file)

In [4]:
import os

image_dir = r"E:\demo dataset\images"
mask_dir = r"E:\demo dataset\binary_masks"

for file in os.listdir(image_dir):
    if not os.path.exists(os.path.join(mask_dir, file)):
        os.remove(os.path.join(image_dir, file))
        print("Deleted:", file)

In [5]:
dataset = LeafDataset(
    image_dir=r"E:\demo dataset\images",
    mask_dir=r"E:\demo dataset\binary_masks"
)

print("Dataset Size:", len(dataset))

Dataset Size: 640


In [6]:
img, target = dataset[0]

print("Image Shape:", img.shape)
print("Boxes:", target["boxes"])
print("Mask Shape:", target["masks"].shape)

Image Shape: torch.Size([3, 256, 256])
Boxes: tensor([[  0.,   0., 255., 255.]])
Mask Shape: torch.Size([1, 256, 256])


In [7]:
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn

model = maskrcnn_resnet50_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features
mask_in_features = model.roi_heads.mask_predictor.conv5_mask.in_channels

model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features, 2
)

model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
    mask_in_features, 256, 2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model Ready on", device)

Model Ready on cuda


In [8]:
from torch.utils.data import DataLoader, random_split

def collate_fn(batch):
    return tuple(zip(*batch))

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

# Training loader
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

# Validation loader
val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print("Train Loader Ready")
print("Validation Loader Ready")

Train Loader Ready
Validation Loader Ready


In [10]:
import torch.optim as optim

optimizer = optim.SGD(
    model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

num_epochs = 25

for epoch in range(num_epochs):

    # ---------------- TRAINING ----------------
    model.train()

    train_loss = 0

    print(f"\n===== Epoch {epoch+1}/{num_epochs} Started =====")

    for images, targets in train_loader:

        images = [img.to(device) for img in images]

        targets = [
            {k: v.to(device) for k, v in t.items()}
            for t in targets
        ]

        loss_dict = model(images, targets)

        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()

        losses.backward()

        optimizer.step()

        train_loss += losses.item()

    # ---------------- VALIDATION ----------------
    model.train()

    val_loss = 0

    with torch.no_grad():

        for images, targets in val_loader:

            images = [img.to(device) for img in images]

            targets = [
                {k: v.to(device) for k, v in t.items()}
                for t in targets
            ]

            loss_dict = model(images, targets)

            losses = sum(loss for loss in loss_dict.values())

            val_loss += losses.item()

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

    scheduler.step()


===== Epoch 1/25 Started =====
Epoch 1 | Train Loss: 97.7596 | Validation Loss: 25.2797

===== Epoch 2/25 Started =====
Epoch 2 | Train Loss: 93.4339 | Validation Loss: 23.8229

===== Epoch 3/25 Started =====
Epoch 3 | Train Loss: 90.1478 | Validation Loss: 23.9485

===== Epoch 4/25 Started =====
Epoch 4 | Train Loss: 89.8186 | Validation Loss: 23.5060

===== Epoch 5/25 Started =====
Epoch 5 | Train Loss: 86.8844 | Validation Loss: 23.8886

===== Epoch 6/25 Started =====
Epoch 6 | Train Loss: 86.1379 | Validation Loss: 23.4912

===== Epoch 7/25 Started =====
Epoch 7 | Train Loss: 83.0277 | Validation Loss: 24.6087

===== Epoch 8/25 Started =====
Epoch 8 | Train Loss: 83.4182 | Validation Loss: 23.6840

===== Epoch 9/25 Started =====
Epoch 9 | Train Loss: 84.0917 | Validation Loss: 22.1383

===== Epoch 10/25 Started =====
Epoch 10 | Train Loss: 83.1356 | Validation Loss: 23.0535

===== Epoch 11/25 Started =====
Epoch 11 | Train Loss: 80.4112 | Validation Loss: 22.7029

===== Epoch 12/2

In [11]:
torch.save(model.state_dict(), "leaf_maskrcnn.pth")
print("Model Saved Successfully")

Model Saved Successfully
